# 🚀 ABSA Master Research Pipeline


Thiết kế chuẩn hóa để Training, Chạy GridSearch Parameters, Đánh giá và Phân tích Lỗi hoàn toàn trên Notebook.


## 1. Import Thư Viện và Core Library


In [1]:
import sys, os
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import clear_output, display

# Trỏ vào thư mục gốc để lấy code (quan trọng)
if '..' not in sys.path: sys.path.append(os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

# 🔁 Import các tính năng cốt lõi từ absa_core.py (đã được bóc tách từ file gốc)
print('Đang nạp dataset và thư viện FastText...')
import training.absa_core as core
print('✅ Đã nạp thành công Core Library!')



Đang nạp dataset và thư viện FastText...
Dependencies installed!
Running on: Local
Device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti
Data dir: .
Save dir: ./dl_improved_v2


AssertionError: File not found: .\train.jsonl. Check DATA_DIR or Kaggle dataset name!

## 2. Xây dững Hàm Thực nghiệm (Experiment Wrapper)
Đóng gói lại quá trình khởi tạo mô hình và training thành một khối duy nhất có khả năng nhận tham số (parameters).


In [ ]:
def run_experiment(config, experiment_id):
    print(f'\n[Exp {experiment_id}] Bắt đầu cấu hình: {config}')
    
    # 1. Load Data Loader với Batch Size mới
    train_loader = torch.utils.data.DataLoader(core.train_ds, batch_size=config['batch_size'], shuffle=True)
    dev_loader = torch.utils.data.DataLoader(core.dev_ds, batch_size=config['batch_size'])
    test_loader = torch.utils.data.DataLoader(core.test_ds, batch_size=config['batch_size'])
    
    # 2. Khởi tạo Model theo Config
    model_type = config['model_type']
    if 'CNN' in model_type:
        model = core.ImprovedCNNCRF(
            vocab_size=core.VOCAB_SIZE, emb_dim=core.EMB_DIM, hidden_dim=config['hidden_dim'],
            num_tags=core.NUM_TAGS, pretrained_emb=core.emb_matrix, pad_idx=core.PAD_IDX,
            dropout=config['dropout'], use_attention=config['use_attention']
        ).to(core.device)
    else:  # Sequence Models
        # Tách tên RNN type (vd: BiLSTM -> lstm, bidir=True)
        is_bidir = 'Bi' in model_type
        rnn_type = model_type.replace('Bi', '').replace('-CRF', '').lower()
        model = core.ImprovedSequenceCRF(
            vocab_size=core.VOCAB_SIZE, emb_dim=core.EMB_DIM, hidden_dim=config['hidden_dim'],
            num_tags=core.NUM_TAGS, pretrained_emb=core.emb_matrix,
            n_layers=config['n_layers'], dropout=config['dropout'], pad_idx=core.PAD_IDX,
            rnn_type=rnn_type, bidir=is_bidir, use_attention=config['use_attention']
        ).to(core.device)
    
    # 3. Setup Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
    
    # 4. Vòng lặp Train Mini (Có thể tinh chỉnh bỏ sớm nếu model ko học được - Early Stopping)
    best_f1, best_state, wait = 0, None, 0
    patience = 5
    
    for ep in range(config['epochs']):
        model.train()
        train_loss = 0
        for b in train_loader:
            optimizer.zero_grad()
            out = model(b['seq'].to(core.device), b['len'], b['mask'].to(core.device), b['tags'].to(core.device))
            out['loss'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += out['loss'].item()
            
        # Eval ở epoch hiện tại
        dev_res = core.predict_all(model, dev_loader)
        dev_f1 = core.f1_score(dev_res['true_sent'].flatten(), dev_res['pred_sent'].flatten(), average='micro')
        scheduler.step()
        
        if dev_f1 > best_f1:
            best_f1 = dev_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'    -> Early stopping at epoch {ep+1}')
                break
                
    # 5. Phục hồi best model và Test
    if best_state: model.load_state_dict(best_state); model.to(core.device)
    test_res = core.predict_all(model, test_loader)
    metrics = core.evaluate_multilabel(test_res['true_sent'], test_res['pred_sent'], core.LABEL_NAMES)
    
    # Đóng gói và trả về kết quả
    return {
        'Exp_ID': experiment_id,
        'Model': model_type,
        'LR': config['lr'],
        'Drop': config['dropout'],
        'Hidden': config['hidden_dim'],
        'Attn': config['use_attention'],
        'F1_Micro': metrics['micro']['f1'],
        'F1_Macro': metrics['macro']['f1'],
        'Tok_Acc': test_res['tok_acc'],
        'Obj_Model': model,           # Giữ model để phân tích Error
        'Test_Outputs': test_res      # Giữ output để inspect Text
    }



## 3. Khởi chạy Chiến Dịch Grid Search\nThay vì chạy code thủ công, tạo Không Gian Thí Nghiệm với itertools.


In [ ]:
# Định nghĩa các tùy chỉnh muốn làm bài kiểm tra:
lrs = [1e-3, 5e-4]
model_types = ['BiLSTM-CRF', 'BiGRU-CRF']
attentions = [True, False]

experiments_grid = []
for lr, m_type, attn in itertools.product(lrs, model_types, attentions):
    experiments_grid.append({
        'model_type': m_type,
        'lr': lr,
        'dropout': 0.3,
        'hidden_dim': 128, # Rút gọn dims cho thử nghiệm nhanh trên Notebook
        'n_layers': 1,
        'batch_size': 64,
        'epochs': 15,      # Rút gọn epoch
        'use_attention': attn
    })

print(f'Mục tiêu chạy: {len(experiments_grid)} Thí nghiệm.')



## 4. Run Pipeline & Export Notebook Dashboard


In [ ]:
all_results = []
models_cache = {}
outputs_cache = {}

for idx, conf in enumerate(experiments_grid):
    clear_output(wait=True) # Xoá màn hình Jupyter cũ
    
    # Hiển thị bảng live cho các exp đã chạy xong
    if len(all_results) > 0:
        display_df = pd.DataFrame(all_results).drop(columns=['Obj_Model', 'Test_Outputs'])
        print('🏆 KẾT QUẢ CÁC MODEL TRƯỚC ĐÓ:')
        display(display_df.style.highlight_max(subset=['F1_Micro', 'F1_Macro'], color='lightgreen'))
        
    print(f'\n[Tiếp tục] Vận hành thí nghiệm {idx+1}/{len(experiments_grid)}...')
    try:
        res = run_experiment(conf, idx+1)
        # Bóc bớt Obj ra vì nó nặng log
        final_res = {k:v for k,v in res.items() if k not in ['Obj_Model', 'Test_Outputs']}
        all_results.append(final_res)
        models_cache[idx+1] = res['Obj_Model']
        outputs_cache[idx+1] = res['Test_Outputs']
    except Exception as e:
        print(f'Error running experiment {idx+1}: {e}')

clear_output(wait=True)
final_df = pd.DataFrame(all_results)
print('🚀 HOÀN TẤT CHIẾN DỊCH TRAINING!')
display(final_df.sort_values(by='F1_Micro', ascending=False).style.highlight_max(subset=['F1_Micro', 'F1_Macro'], color='lightgreen'))



## 5. Visualization Báo Cáo\nVẽ biểu đồ F1-Score so sánh giữa các thuật toán và Setting giúp báo cáo thêm chuyên nghiệp.


In [ ]:
# Vẽ biểu đồ So sánh Micro F1
plt.figure(figsize=(10, 6))
sns.barplot(data=final_df, x='Model', y='F1_Micro', hue='Attn', ci=None)
plt.title('So Sánh F1 Micro-Score giữa các Mô Hình (Có/Không có Attention)', fontsize=14)
plt.ylim(0.4, 0.9) # Tùy vào range thực tế
plt.show()

# Heatmap độ tương quan
pivot_df = final_df.pivot_table(index='Model', columns='LR', values='F1_Macro', aggfunc='mean')
plt.figure(figsize=(8, 4))
sns.heatmap(pivot_df, annot=True, cmap='Blues', fmt='.4f')
plt.title('Tác động của Learning Rate lên Macro F1-Score')
plt.show()



## 6. Error Analysis (Phân Tích Lỗi) - Phần lấy Điểm Nghiên Cứu Cao\nBóc tách những câu đoán sai trong Model Tốt Nhất.


In [ ]:
best_exp_id = final_df.loc[final_df['F1_Micro'].idxmax()]['Exp_ID']
print(f'Mô hình tốt nhất là Experiment #{best_exp_id}. Bắt đầu trích xuất lỗi...')

test_out = outputs_cache[best_exp_id]
true_sent = test_out['true_sent']
pred_sent = test_out['pred_sent']

error_records = []
# Lặp qua tập test. core.test_items là tập dữ liệu dạng original list dictionary
# Lưu ý: cần khớp index. test_out outputs map trực tiếp tới core.test_items
for i in range(len(core.test_items)):
    if not np.array_equal(true_sent[i], pred_sent[i]):
        # Extract wrong labels
        true_labels = [core.LABEL_NAMES[j] for j, val in enumerate(true_sent[i]) if val == 1.0]
        pred_labels = [core.LABEL_NAMES[j] for j, val in enumerate(pred_sent[i]) if val == 1.0]
        
        error_records.append({
            'Text': core.test_items[i]['text'],
            'True_Labels': ', '.join(true_labels) if true_labels else '[]',
            'Predicted': ', '.join(pred_labels) if pred_labels else '[]'
        })

err_df = pd.DataFrame(error_records)
print(f'\nTổng số câu đoán sai hoàn toàn hoặc sai 1 phần (False Positives / Negatives): {len(err_df)} / {len(core.test_items)}')

# Show 10 lỗi điển hình:
display(err_df.sample(10) if len(err_df) >= 10 else err_df)

# Phân tích xem nhãn nào bị sai nhiều nhất (Ví dụ BATTERY#POSITIVE vs BATTERY#NEUTRAL dễ nhầm nhau)
# (Bạn có thể code confusion matrix tại đây)

